# IT 520 — Does ZIP help this file?
**Friday, October 16, 2026 · Individual, asynchronous · About 40 minutes**

Complete this in **Google Colab** and submit your completed notebook to **Canvas by Saturday** (use the deadline shown in Canvas).

1. Open this `.ipynb` in Colab using **File → Upload notebook**, then choose **File → Save a copy in Drive**. Give the copy your name.
2. Run cells in order using the ▶ button beside each code cell. Wait for each to finish. The supplied code handles the mechanics. The only code you change is typing your Q2 numbers into one cell.
3. Double-click each **Your answer** text cell, replace its placeholders, then press Shift+Enter. **Make your predictions before running the measurement cell.**
4. At the end, choose **File → Download → Download .ipynb** and upload that file to Canvas. Your written answers and displayed results must both be present. Reopen the downloaded notebook to check them before submitting.

If Colab disconnects, reconnect and rerun the code cells in order; your written answers in your saved copy remain. Do not use “Run all” until you have recorded your predictions.

Today you will distinguish **smaller**, **exactly recovered**, and **faster**. We measure the first two. We do not measure speed. All four original files are included in this notebook: no external file downloads, package installs, or Drive mounting are needed.

## 1. Inspect the inputs
Run this cell. Read the text samples and view the photograph. The log preview shows only its first six lines; the actual file contains six hours of readings.

The scanner photograph is by Rodrigo César, Wikimedia Commons, public domain: [source](https://commons.wikimedia.org/wiki/File:Barcode-hand-scanner.jpg). It is the same image used in the September 23 lesson.

In [ ]:
#@title Supplied inputs: run this cell (▶). Nothing to edit; the code is hidden because it contains the photo data.
import base64
import hashlib
import platform
import zlib
from pathlib import Path
from IPython.display import display, Image

WORK = Path("it520_zip_activity")
ORIGINALS = WORK / "originals"
ARCHIVES = WORK / "archives"
RECOVERED = WORK / "recovered"
for folder in (ORIGINALS, ARCHIVES, RECOVERED):
    folder.mkdir(parents=True, exist_ok=True)

lines = []
for i in range(6 * 60 * 3):
    total = 3 * 3600 + i * 20
    h, m, s = total // 3600, total // 60 % 60, total % 60
    sensor = f"sensor-{i % 4 + 1:02d}"
    status = "WARN temp high" if i == 1000 else "OK"
    lines.append(f"2026-10-12 {h:02d}:{m:02d}:{s:02d} gateway-3 {sensor} status={status}\n")
ESSAY = 'The Town Library Moves Across the Street\n\nFor forty-one years the town library sat in the old brick bank on the corner of Main and Pleasant. The vault became the local history room, the teller windows became the checkout desk, and the manager\'s office, with its tall arched window, became the place where the children\'s librarian read stories every Tuesday morning. Nobody planned it that way. The town bought the building cheaply after the bank closed, and the library simply grew into whatever shape the rooms allowed.\n\nBy last spring the building had run out of patience. The roof leaked over the periodicals, the furnace needed a part that no company still made, and the single narrow staircase meant that anyone who could not climb stairs never saw the second floor at all. When the hardware store across the street closed, the trustees made an offer, and in June the town voted to buy it. The new building was one large open floor with a loading dock in the back. It was plain, but it was dry, warm, and flat.\n\nMoving a library turns out to be less about lifting boxes than about keeping order. Thirty-eight thousand items had to leave one building and arrive in another in a sequence that would let the shelves be filled from left to right without anyone having to reshelve a whole wall twice. The head librarian, a careful woman named Dolores Fenn, spent most of July with a clipboard and a tape measure. She measured every shelf in the old building and every bay in the new one, and she wrote the numbers on index cards that she kept in a shoebox on her desk.\n\nVolunteers came from everywhere. The high school soccer team carried boxes for three Saturdays in a row. A retired engineer built rolling carts out of plywood and casters that he found in his garage. Two sisters who had not spoken to each other in years ended up assigned to the same table, labeling boxes, and by the end of the second week they were bringing each other coffee. People who had not used the library since they were children came in to help and left with library cards.\n\nThere were mistakes, of course. One afternoon a cart of large-print mysteries rolled down the ramp at the loading dock faster than anyone expected and spilled across the parking lot. A box marked "Local History, Do Not Open" turned out to contain the holiday decorations. The new shelves for picture books arrived a foot taller than ordered, so the smallest readers could not reach the top rows until the engineer cut them down with a borrowed saw.\n\nThe hardest part was the local history room. The old vault held town reports going back to 1803, maps drawn by hand, and a set of glass photographic plates that showed Main Street before it was paved. None of it could be replaced. Dolores refused to let those items ride in the volunteer carts. Instead she and the town clerk carried them across the street themselves, one box at a time, early on a Sunday morning when there was no traffic. It took them six hours.\n\nThe new library opened on the second Saturday in September. There was a ribbon, a short speech by the town manager, and a cake shaped like a book that was, everyone agreed, slightly lopsided. The children\'s corner now sits beside a wide window that faces the street, and on Tuesday mornings people walking past can see the story circle through the glass. The second floor is gone, which means everyone can reach everything.\n\nThe old bank is for sale again. Some people hope it will become a restaurant. Others say it should stay empty for a while, out of respect. Dolores says she does not mind either way. She kept one thing from the old building: the brass sign that read "Please Wait Here for the Next Available Teller," which now hangs, without explanation, above the new checkout desk.\n'
PHOTO_BASE64 = '...'

inputs = {
    "server-log.txt": "".join(lines).encode("ascii"),
    "essay.txt": ESSAY.encode("utf-8"),
    "photo.jpg": base64.b64decode(PHOTO_BASE64),
    "note.txt": b"Meeting moved to 3pm. Bring the notes.\n",
}
PINS = {'server-log.txt': '106472d60a8e97027eced320bf213732aa811b0fe7afef0ede02f1de2467e1fa', 'photo.jpg': '28c13095093cede3f109ea5f0f8a4a5ddc475c3de0a71484d458100bb961cd7f', 'essay.txt': '7e0f9e65b8a53c8df97af2acdb97d7fbf85ad041a478ed3f016e9f5a640741ac', 'note.txt': '8208d404ab41332bd79dc46becab83647a19f125d5e14a66d1402092a3245c98'}

for name, data in inputs.items():
    if hashlib.sha256(data).hexdigest() != PINS[name]:
        raise ValueError(f"Input check failed for {name}. Reopen the original notebook.")
    (ORIGINALS / name).write_bytes(data)

print("Log preview (first six lines):\n" + "".join(lines[:6]))
print("Prose:\n" + ESSAY)
print("Tiny note:\n" + inputs["note.txt"].decode())
display(Image(data=inputs["photo.jpg"], width=300))
print("Inputs ready. Record your predictions below before continuing.")

### Your answer — Q1: predict before measuring
Rank all four files from greatest to least **percentage saved**, not total bytes. Could any grow? Explain your prediction using what you saw inside each file.

**My ranking:** [replace]

**My reasons:** [replace]

**File(s) that might grow, and why:** [replace]

## 2. Compress separately and measure
Run the next cell only after Q1. It creates **one ZIP per file**, using Python's `zipfile` with DEFLATE at level 6. It reports the original length and the **entire archive length**, including the ZIP's bookkeeping. It overwrites only the activity's own archives, so rerunning is safe.

For Q2 you will calculate two values for each file in the code cell after this one:

- **Ratio = ZIP bytes ÷ original bytes** (round to 3 decimal places).
- **Percentage saved = (original bytes − ZIP bytes) ÷ original bytes × 100** (round to 1 decimal place).

A negative percentage saved means growth. These are byte counts, not rounded KB or disk allocation sizes.

In [ ]:
# Run after writing Q1. Nothing to edit.
import zipfile

measurements = {}
print(f"Environment: Python {platform.python_version()}, {platform.system()}, zlib {zlib.ZLIB_RUNTIME_VERSION}")
print("Tool: Python zipfile; ZIP_DEFLATED; compresslevel=6; one entry per archive")
print(f"{'File':<18} {'Original bytes':>16} {'Whole ZIP bytes':>18}")
for name in inputs:
    archive = ARCHIVES / (name + ".zip")
    with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zipped:
        zipped.write(ORIGINALS / name, arcname=name)
    original_size = (ORIGINALS / name).stat().st_size
    zip_size = archive.stat().st_size
    measurements[name] = (original_size, zip_size)
    print(f"{name:<18} {original_size:>16,} {zip_size:>18,}")

def check_q2(my_ratio, my_saved):
    """Compare your Q2 arithmetic with the measured bytes above."""
    all_ok = True
    for name, (original, zipped) in measurements.items():
        ratio, saved = zipped / original, (original - zipped) / original * 100
        r, s = my_ratio.get(name), my_saved.get(name)
        if r is None or s is None:
            print(f"{name:<16} fill in both values"); all_ok = False; continue
        if not all(isinstance(v, (int, float)) for v in (r, s)):
            print(f"{name:<16} use numbers, not text in quotes"); all_ok = False; continue
        notes = []
        if abs(r - ratio) > 0.001:
            notes.append("ratio upside down? use ZIP bytes / original bytes"
                         if abs(r - original / zipped) < 0.01 else "ratio: check the division")
        if abs(s - saved) > 0.1:
            notes.append("% saved: the sign is wrong; growth is negative"
                         if abs(s + saved) <= 0.1 else "% saved: check the arithmetic")
        all_ok = all_ok and not notes
        print(f"{name:<16} " + ("OK" if not notes else "; ".join(notes)))
    print("\nQ2 arithmetic: PASS" if all_ok else "\nFix the rows above, then run this cell again.")


In [ ]:
# Q2. Replace each None with your calculation, then run this cell.
# You may type the arithmetic itself, for example  1234 / 5678  (use YOUR measured bytes).
my_ratio = {           # ZIP bytes / original bytes
    "server-log.txt": None,
    "essay.txt": None,
    "photo.jpg": None,
    "note.txt": None,
}
my_percent_saved = {   # (original - ZIP) / original * 100; negative means it grew
    "server-log.txt": None,
    "essay.txt": None,
    "photo.jpg": None,
    "note.txt": None,
}
check_q2(my_ratio, my_percent_saved)

### Your answers — Q2–Q3
**Q2.** Your calculations are in the code cell above; keep running it until it says **PASS**. Then show the arithmetic for one row here, using your own measured bytes.

**One worked calculation:** [replace]

**Q3.** Record the environment and compression tool/settings printed above. These describe the Colab runtime, which may differ from your laptop.

[replace]

## 3. Recover and test
Run this cell. It extracts the log and photo into a **separate recovered folder**, then compares each extracted file with its original **byte for byte**. Read both paths: comparing the original to itself proves nothing.

It then makes a deliberately changed copy of the recovered log, changing one byte while keeping its length. Predict its comparison result before running.

**My prediction for the changed copy:** [replace]

In [ ]:
# Extract two files and compare every byte. Nothing to edit.
def compare_files(first, second):
    first, second = Path(first), Path(second)
    if first.resolve() == second.resolve():
        raise ValueError("Choose two different paths, not the same file twice.")
    a, b = first.read_bytes(), second.read_bytes()
    verdict = "IDENTICAL" if a == b else "DIFFERENT"
    print(f"{first} ({len(a):,} bytes)\n{second} ({len(b):,} bytes)\n{verdict}\n")
    return verdict

recovery_results = {}
for name in ("server-log.txt", "photo.jpg"):
    with zipfile.ZipFile(ARCHIVES / (name + ".zip")) as zipped:
        if zipped.namelist() != [name]:
            raise ValueError("Unexpected archive contents. Rerun the compression cell.")
        zipped.extract(name, RECOVERED)
    recovery_results[name] = compare_files(ORIGINALS / name, RECOVERED / name)

changed = bytearray((RECOVERED / "server-log.txt").read_bytes())
changed[0] ^= 1
changed_path = RECOVERED / "changed-log.txt"
changed_path.write_bytes(changed)
changed_result = compare_files(ORIGINALS / "server-log.txt", changed_path)

### Your answers — Q4–Q5
**Q4.** For each recovery check, record **both paths** and the verdict. Then record the changed-copy verdict. If recovery says DIFFERENT, rerun the input, compression, and recovery cells in order and describe what happened.

- Log original / recovered paths and result: [replace]
- Photo original / recovered paths and result: [replace]
- Changed log paths and result: [replace]

**Q5.** Use the changed-copy experiment to explain why equal byte counts do not establish exact recovery. Why would opening both photographs and deciding they look the same also be insufficient?

[replace]

We checked recovery for **two** originals. Do not claim recovery was verified for the other two.

## 4. Explain and recommend — your answers
**Q6.** Which file had the greatest percentage saved? Explain using its **contents**, not only its extension. Compare it with the other text file.

[replace]

**Q7.** Which file(s) grew or barely shrank? Explain each case. What must a ZIP store besides the compressed contents so it can give the file back? Does growing imply data loss? Cite your recovery evidence where applicable.

[replace]

**Q8.** Recommend which files you would ZIP to save storage or bytes sent, citing measurements. State your criterion: for example, any byte saving versus a saving large enough to justify added work. Why do these measurements **not** show that sending the files would be faster? What else would you need to measure?

[replace]

## 5. Exit — your answer
**Q9.** Which predictions were supported or overturned? Cite at least **two numbers** from your table and name one limit on what your experiment establishes.

[replace]

## Optional extension — not required
Suppose ZIP creation plus extraction of `server-log.txt` takes a combined **t = 0.01 seconds**. This is hypothetical; you did not measure it.

With your measured original bytes **O**, ZIP bytes **C**, and link speed **s** in bytes/second:

- Plain route: O/s seconds.
- Zipped route: C/s + t seconds.

At what speed are they equal? Which route wins below and above that speed? Show units. What happens for `note.txt`?

**Optional answer:** [replace]

## Submit
Check that Q1–Q9 are filled in, your Q2 check shows PASS, and the measurement and comparison outputs are visible. Save your Drive copy, download the completed `.ipynb` using **File → Download**, and upload it to Canvas. Reopen your downloaded file to confirm your answers and outputs survived. Submit the notebook, not the four ZIP files.